In [1]:
# --- Cell 1: Imports and setup ---
import os
import numpy as np
import scipy.io as sio
import trimesh
import k3d

In [2]:
# --- Cell 2: Paths and file listing ---
BASE_DIR = "dropbox/Chair"

models_dir = os.path.join(BASE_DIR, "models")
labels_dir = os.path.join(BASE_DIR, "labels")
boxes_dir = os.path.join(BASE_DIR, "boxes")
ops_dir = os.path.join(BASE_DIR, "ops")
syms_dir = os.path.join(BASE_DIR, "syms")
parts_dir = os.path.join(BASE_DIR, "part mesh indices")
obbs_dir = os.path.join(BASE_DIR, "obbs")

# Print folder contents to verify
print("Models:", os.listdir(models_dir)[:5])
print("Labels:", os.listdir(labels_dir)[:5])
print("Boxes:", os.listdir(boxes_dir)[:5])
print("Ops:", os.listdir(ops_dir)[:5])
print("Syms:", os.listdir(syms_dir)[:5])
print("Part Mesh Indices:", os.listdir(parts_dir)[:5])
print("OBBs:", os.listdir(obbs_dir)[:5])

Models: ['2819.obj', '43912.obj', '39066.obj', '38378.obj', '36433.obj']
Labels: ['259.mat', '1915.mat', '3864.mat', '3870.mat', '1901.mat']
Boxes: ['259.mat', '1915.mat', '3864.mat', '3870.mat', '1901.mat']
Ops: ['259.mat', '1915.mat', '3864.mat', '3870.mat', '1901.mat']
Syms: ['259.mat', '1915.mat', '3864.mat', '3870.mat', '1901.mat']
Part Mesh Indices: ['259.mat', '1915.mat', '3864.mat', '3870.mat', '1901.mat']
OBBs: ['2629.obb', '40559.obb', '37909.obb', '35878.obb', '3251.obb']


In [3]:
# --- Cell 3: Load a mesh and sample points ---
mesh_file = "2819.obj"  # pick a model from the Models folder
mesh_path = os.path.join(models_dir, mesh_file)

# Load the mesh using trimesh
mesh = trimesh.load(mesh_path)
print(mesh)

# Sample 2048 points on the mesh surface
points, _ = trimesh.sample.sample_surface(mesh, 2048)
print("Sampled points shape:", points.shape)  # should be (2048, 3)

<trimesh.Trimesh(vertices.shape=(126153, 3), faces.shape=(252447, 3))>
Sampled points shape: (2048, 3)


In [4]:
# --- Cell 4: Load part labels (.mat) ---
label_file = "259.mat"  # choose the corresponding label file
label_path = os.path.join(labels_dir, label_file)

labels_mat = sio.loadmat(label_path)

# Find valid keys (ignore metadata)
valid_keys = [k for k in labels_mat.keys() if not k.startswith("__")]
print("Valid keys in .mat file:", valid_keys)

# Pick the first valid key
label_key = valid_keys[0]
labels = labels_mat[label_key].squeeze()
print("Labels shape:", labels.shape)
print("Unique part IDs:", np.unique(labels))

Valid keys in .mat file: ['label']
Labels shape: (7,)
Unique part IDs: [0 1 2]


In [5]:
# --- Cell 5: k3d visualization (temporary mapping for small label array) ---
import k3d

# Map sampled points to labels (simple repeat to match 2048 points)
labels_per_point = np.tile(labels, int(np.ceil(points.shape[0]/labels.shape[0])))[:points.shape[0]].astype(int)

# Assign a color per unique label
unique_labels = np.unique(labels_per_point)
colors_hex = np.random.randint(0, 0xFFFFFF, size=unique_labels.max() + 1, dtype=np.uint32)
point_colors = colors_hex[labels_per_point]

# Create k3d plot
plot = k3d.plot(grid_visible=True, background_color=0xEEEEEE)
points_plot = k3d.points(
    positions=points.astype(np.float32),
    colors=point_colors,
    point_size=0.03
)
plot += points_plot
plot.display()

/opt/homebrew/Caskroom/miniconda/base/envs/abs/lib/python3.12/site-packages/traittypes/traittypes.py:97: UserWarning: Given trait value dtype "float32" does not match required type "float32". A coerced copy has been created.
  warnings.warn(


Output()

In [6]:
# --- Cell 6: Generate per-point labels from part mesh indices (fixed for MATLAB cell arrays) ---
part_idx_file = "259.mat"
part_idx_path = os.path.join(parts_dir, part_idx_file)

part_idx_mat = sio.loadmat(part_idx_path)
valid_keys = [k for k in part_idx_mat.keys() if not k.startswith("__")]
print("Keys in part mesh indices:", valid_keys)

part_idx_key = valid_keys[0]
part_indices_cell = part_idx_mat[part_idx_key]  # this is a NumPy object array (cell array)

mesh_vertices = mesh.vertices
labels_per_point = np.zeros(points.shape[0], dtype=int)

for part_id in range(part_indices_cell.shape[0]):
    idx_array = part_indices_cell[part_id, 0]  # extract the array from the cell
    if idx_array.size == 0:
        continue
    idx_array = idx_array.astype(int).flatten() - 1  # convert to Python 0-based
    part_vertices = mesh_vertices[idx_array]
    
    # Compute distances to sampled points
    dists = np.linalg.norm(points[:, None, :] - part_vertices[None, :, :], axis=2)
    nearest = np.argmin(dists, axis=1)
    labels_per_point[nearest] = part_id

print("Per-point labels assigned:", labels_per_point.shape)
print("Unique labels:", np.unique(labels_per_point))

Keys in part mesh indices: ['cell_boxs_correspond_objSerialNumber', 'shapename']
Per-point labels assigned: (2048,)
Unique labels: [0]


In [7]:
# --- Cell 7: k3d visualization with random part labels for demonstration ---
import k3d

# Assign random part IDs (for visualization only)
num_parts = 3  # for example, based on your 'Unique part IDs: [0 1 2]'
labels_per_point = np.random.randint(0, num_parts, size=points.shape[0])

# Assign colors per label
unique_labels = np.unique(labels_per_point)
colors_hex = np.random.randint(0, 0xFFFFFF, size=unique_labels.max() + 1, dtype=np.uint32)
point_colors = colors_hex[labels_per_point]

# Create k3d plot
plot = k3d.plot(grid_visible=True, background_color=0xEEEEEE)
points_plot = k3d.points(
    positions=points.astype(np.float32),
    colors=point_colors,
    point_size=0.03
)
plot += points_plot
plot.display()

Output()

In [8]:
# --- Cell 8: Inspect ops.mat and syms.mat (fixed for 1D sym array) ---
ops_file = "259.mat"
ops_path = os.path.join(ops_dir, ops_file)
ops_mat = sio.loadmat(ops_path)

ops_keys = [k for k in ops_mat.keys() if not k.startswith("__")]
print("Keys in ops.mat:", ops_keys)
ops_data = ops_mat[ops_keys[0]]
print("Ops shape:", ops_data.shape)
print("Ops array (first 10 entries):", ops_data[:, :10])  # show first 10 columns

# Load syms
syms_file = "259.mat"
syms_path = os.path.join(syms_dir, syms_file)
syms_mat = sio.loadmat(syms_path)

syms_keys = [k for k in syms_mat.keys() if not k.startswith("__")]
print("Keys in syms.mat:", syms_keys)
syms_data = syms_mat[syms_keys[0]]
print("Syms array type:", type(syms_data), "length:", len(syms_data))

# Show first 5 elements of the 1D array
for i, elem in enumerate(syms_data[:5]):
    print(f"Element {i}: type {type(elem)}, shape {elem.shape if hasattr(elem, 'shape') else 'N/A'}")

Keys in ops.mat: ['op']
Ops shape: (1, 16)
Ops array (first 10 entries): [[0 0 0 1 2 1 0 0 1 2]]
Keys in syms.mat: ['shapename', 'sym']
Syms array type: <class 'numpy.ndarray'> length: 1
Element 0: type <class 'numpy.str_'>, shape ()


In [9]:
# --- Cell 9: Build readable part hierarchy from ops ---
ops_flat = ops_data.flatten()
node_types = {0: "LEAF", 1: "ADJ", 2: "SYM"}  # map ops codes to type names

tree_stack = []
print("Building tree from ops...")

for i, op in enumerate(ops_flat):
    op_name = node_types.get(op, f"UNKNOWN({op})")
    
    if op_name == "LEAF":
        tree_stack.append(f"LEAF_{i}")
    elif op_name == "ADJ":
        if len(tree_stack) >= 2:
            right = tree_stack.pop()
            left = tree_stack.pop()
            tree_stack.append(f"ADJ({left}, {right})")
        else:
            tree_stack.append("ADJ(??)")
    elif op_name == "SYM":
        if len(tree_stack) >= 1:
            child = tree_stack.pop()
            tree_stack.append(f"SYM({child})")
        else:
            tree_stack.append("SYM(??)")
    else:
        tree_stack.append(f"NODE_{i}")

print("Final tree representation:")
print(tree_stack[0] if tree_stack else "Empty tree")

Building tree from ops...
Final tree representation:
ADJ(ADJ(LEAF_0, SYM(ADJ(LEAF_1, LEAF_2))), ADJ(SYM(ADJ(LEAF_6, LEAF_7)), ADJ(LEAF_10, SYM(LEAF_11))))


In [10]:
# --- Cell 10: k3d visualization by subtree hierarchy ---
import random

num_points = points.shape[0]
ops_flat = ops_data.flatten()

# Assign a random color to each LEAF
leaf_indices = [i for i, op in enumerate(ops_flat) if op == 0]
colors_hex = np.random.randint(0, 0xFFFFFF, size=len(leaf_indices), dtype=np.uint32)

# Map points to LEAFs (for demo, evenly split points)
labels_per_point = np.zeros(num_points, dtype=int)
points_per_leaf = num_points // len(leaf_indices)
for i, leaf_id in enumerate(leaf_indices):
    start = i * points_per_leaf
    end = start + points_per_leaf if i != len(leaf_indices) - 1 else num_points
    labels_per_point[start:end] = i

# Map colors
point_colors = colors_hex[labels_per_point]

# k3d plot
plot = k3d.plot(grid_visible=True, background_color=0xEEEEEE)
points_plot = k3d.points(
    positions=points.astype(np.float32),
    colors=point_colors,
    point_size=0.03
)
plot += points_plot
plot.display()

Output()

In [11]:
# --- Cell 11: k3d visualization with subtree coloring ---
import re

# Extract LEAF indices from the tree string
tree_str = "ADJ(ADJ(LEAF_0, SYM(ADJ(LEAF_1, LEAF_2))), ADJ(SYM(ADJ(LEAF_6, LEAF_7)), ADJ(LEAF_10, SYM(LEAF_11))))"

# Define subtrees manually for demonstration
subtrees = [
    ["LEAF_0", "LEAF_1", "LEAF_2"],   # left subtree
    ["LEAF_6", "LEAF_7", "LEAF_10", "LEAF_11"]  # right subtree
]

num_points = points.shape[0]
labels_per_point = np.zeros(num_points, dtype=int)

# Map points to subtrees (rough equal split)
points_per_leaf = num_points // (len(subtrees[0]) + len(subtrees[1]))
start = 0
for group_id, leaves in enumerate(subtrees):
    for leaf in leaves:
        end = start + points_per_leaf
        labels_per_point[start:end] = group_id
        start = end
# Any remaining points go to last group
labels_per_point[start:] = len(subtrees) - 1

# Assign a random color per subtree
num_groups = len(subtrees)
colors_hex = np.random.randint(0, 0xFFFFFF, size=num_groups, dtype=np.uint32)
point_colors = colors_hex[labels_per_point]

# k3d plot
plot = k3d.plot(grid_visible=True, background_color=0xEEEEEE)
points_plot = k3d.points(
    positions=points.astype(np.float32),
    colors=point_colors,
    point_size=0.03
)
plot += points_plot
plot.display()

Output()

In [12]:
# --- Cell 12: k3d visualization with bounding boxes (correct indices) ---
import k3d
import numpy as np

def cube_geometry(size):
    """
    Returns vertices and triangle indices for a cube of given size (dx, dy, dz) centered at origin.
    """
    dx, dy, dz = size / 2
    vertices = np.array([
        [-dx, -dy, -dz],
        [ dx, -dy, -dz],
        [ dx,  dy, -dz],
        [-dx,  dy, -dz],
        [-dx, -dy,  dz],
        [ dx, -dy,  dz],
        [ dx,  dy,  dz],
        [-dx,  dy,  dz],
    ], dtype=np.float32)
    
    # 12 triangles (two per face)
    indices = np.array([
        [0,1,2],[0,2,3],
        [4,5,6],[4,6,7],
        [0,1,5],[0,5,4],
        [2,3,7],[2,7,6],
        [1,2,6],[1,6,5],
        [0,3,7],[0,7,4]
    ], dtype=np.uint32)
    
    return vertices, indices

# --- plot ---
plot = k3d.plot(grid_visible=True, background_color=0xEEEEEE)

# Add point cloud
points_plot = k3d.points(
    positions=points.astype(np.float32),
    color=0xAAAAAA,
    point_size=0.03
)
plot += points_plot

# Add bounding boxes (demo 5 cubes)
for i in range(5):
    center = np.random.rand(3)
    size = np.array([0.1, 0.1, 0.1])
    vertices, indices = cube_geometry(size)
    vertices += center  # translate to center
    box_mesh = k3d.mesh(
        vertices=vertices,
        indices=indices,
        color=0xFF0000,
        wireframe=True
    )
    plot += box_mesh

plot.display()

Output()

In [13]:
# --- Cell 13: Load .obb and visualize oriented bounding boxes ---
import os

def obb_to_vertices(center, axes, extents):
    """
    Convert OBB to 8 corner vertices
    center: (3,)
    axes: (3,3) each column is an axis vector
    extents: (3,) half-lengths along axes
    """
    corners = []
    for dx in [-0.5, 0.5]:
        for dy in [-0.5, 0.5]:
            for dz in [-0.5, 0.5]:
                corner = center + dx*extents[0]*axes[:,0] + dy*extents[1]*axes[:,1] + dz*extents[2]*axes[:,2]
                corners.append(corner)
    return np.array(corners, dtype=np.float32)

def cube_indices():
    return np.array([
        [0,1,2],[0,2,3],
        [4,5,6],[4,6,7],
        [0,1,5],[0,5,4],
        [2,3,7],[2,7,6],
        [1,2,6],[1,6,5],
        [0,3,7],[0,7,4]
    ], dtype=np.uint32)

# Example: visualize first 5 .obb files
plot = k3d.plot(grid_visible=True, background_color=0xEEEEEE)
plot += k3d.points(positions=points.astype(np.float32), color=0xAAAAAA, point_size=0.03)

for i, obb_file in enumerate(os.listdir(obbs_dir)[:5]):
    obb_path = os.path.join(obbs_dir, obb_file)
    # Load obb data: center, axes, extents
    obb_data = np.loadtxt(obb_path)
    center = obb_data[:3]
    axes = obb_data[3:12].reshape(3,3)
    extents = obb_data[12:15]
    
    vertices = obb_to_vertices(center, axes, extents)
    indices = cube_indices()
    
    box_mesh = k3d.mesh(vertices=vertices, indices=indices, color=0xFF0000, wireframe=True)
    plot += box_mesh

plot.display()

ValueError: could not convert string 'N' to float64 at row 0, column 1.

In [14]:
# --- Cell 14: Print first .obb file ---
obb_file = sorted(os.listdir(obbs_dir))[0]
obb_path = os.path.join(obbs_dir, obb_file)

with open(obb_path, 'r') as f:
    lines = f.readlines()

print(f"Contents of {obb_file}:\n")
for line in lines:
    print(line.strip())

Contents of 1095.obb:

N 3
0.0302842 0.488309 -0.315397 -0.995024 0.0971271 -0.0222212 -0.00601635 0.164046 0.986434 0.0994548 0.981659 -0.162646 0.89214 0.305279 0.781198
0.00419755 -0.422335 0.169623 0.999916 0.0121707 -0.00438424 -0.0121707 0.770206 -0.637679 -0.00438424 0.637679 0.77029 0.898086 0.523685 1.15351
0.0030865 0.148677 0.0724135 1 0 0 0 1 0 0 0 1 0.890567 0.223845 0.805681
C 2
0 2
1 2
S 0
L 3
0
2
1


In [15]:
# --- Cell 15: Parse numeric OBBs and visualize in k3d ---
import os
import k3d
import numpy as np

def parse_numeric_obb_lines(path):
    """
    Parse a PartNet .obb file: only take numeric lines after headers like 'N', 'C', etc.
    Returns a list of tuples: (center, axes, extents)
    """
    with open(path, 'r') as f:
        lines = f.readlines()
    
    obb_list = []
    for line in lines:
        parts = line.strip().split()
        # Skip non-numeric lines
        try:
            nums = [float(p) for p in parts]
        except:
            continue
        if len(nums) >= 15:  # likely a center+axes+extents line
            center = np.array(nums[:3])
            axes = np.array(nums[3:12]).reshape(3,3)
            extents = np.array(nums[12:15])
            obb_list.append((center, axes, extents))
    return obb_list

def obb_to_vertices(center, axes, extents):
    """
    Convert OBB to 8 corner vertices
    """
    corners = []
    for dx in [-0.5, 0.5]:
        for dy in [-0.5, 0.5]:
            for dz in [-0.5, 0.5]:
                corner = center + dx*extents[0]*axes[:,0] + dy*extents[1]*axes[:,1] + dz*extents[2]*axes[:,2]
                corners.append(corner)
    return np.array(corners, dtype=np.float32)

def cube_indices():
    return np.array([
        [0,1,2],[0,2,3],
        [4,5,6],[4,6,7],
        [0,1,5],[0,5,4],
        [2,3,7],[2,7,6],
        [1,2,6],[1,6,5],
        [0,3,7],[0,7,4]
    ], dtype=np.uint32)

# --- Plot point cloud + OBBs ---
plot = k3d.plot(grid_visible=True, background_color=0xEEEEEE)
plot += k3d.points(positions=points.astype(np.float32), color=0xAAAAAA, point_size=0.03)

# Example: first 5 .obb files
for obb_file in sorted(os.listdir(obbs_dir))[:5]:
    obb_path = os.path.join(obbs_dir, obb_file)
    obb_list = parse_numeric_obb_lines(obb_path)
    
    for center, axes, extents in obb_list:
        vertices = obb_to_vertices(center, axes, extents)
        indices = cube_indices()
        box_mesh = k3d.mesh(vertices=vertices, indices=indices, color=0xFF0000, wireframe=True)
        plot += box_mesh

plot.display()

Output()

In [16]:
# --- Cell 16: Visualize one .obb file + point cloud ---
import os
import k3d
import numpy as np

# Pick the first .obb file
obb_file = sorted(os.listdir(obbs_dir))[0]
obb_path = os.path.join(obbs_dir, obb_file)

def parse_numeric_obb_lines(path):
    """
    Parse numeric OBB lines from a PartNet .obb file
    """
    with open(path, 'r') as f:
        lines = f.readlines()
    
    obb_list = []
    for line in lines:
        parts = line.strip().split()
        try:
            nums = [float(p) for p in parts]
        except:
            continue
        if len(nums) >= 15:  # likely a center+axes+extents line
            center = np.array(nums[:3])
            axes = np.array(nums[3:12]).reshape(3,3)
            extents = np.array(nums[12:15])
            obb_list.append((center, axes, extents))
    return obb_list

def obb_to_vertices(center, axes, extents):
    corners = []
    for dx in [-0.5,0.5]:
        for dy in [-0.5,0.5]:
            for dz in [-0.5,0.5]:
                corner = center + dx*extents[0]*axes[:,0] + dy*extents[1]*axes[:,1] + dz*extents[2]*axes[:,2]
                corners.append(corner)
    return np.array(corners, dtype=np.float32)

def cube_indices():
    return np.array([
        [0,1,2],[0,2,3],
        [4,5,6],[4,6,7],
        [0,1,5],[0,5,4],
        [2,3,7],[2,7,6],
        [1,2,6],[1,6,5],
        [0,3,7],[0,7,4]
    ], dtype=np.uint32)

# --- Plot ---
plot = k3d.plot(grid_visible=True, background_color=0xEEEEEE)
plot += k3d.points(positions=points.astype(np.float32), color=0xAAAAAA, point_size=0.03)

obb_list = parse_numeric_obb_lines(obb_path)
for center, axes, extents in obb_list:
    vertices = obb_to_vertices(center, axes, extents)
    indices = cube_indices()
    box_mesh = k3d.mesh(vertices=vertices, indices=indices, color=0xFF0000, wireframe=True)
    plot += box_mesh

plot.display()

Output()